# FairFace Model Evaluation on MSKCC Dataset (v2)

Toggle `EVAL_MODE` between `'3-way'` and `'6-way'` in Cell 1.

**Only clinical close-up images** — dermoscopic images are excluded
because polarised light alters perceived skin tone.

In [ ]:
# ==================== CELL 1: Configuration ====================
import os, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import label_binarize
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Prevent PIL from crashing on truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- TOGGLE THIS ---
EVAL_MODE = '3-way'   # '3-way' or '6-way'
NUM_CLASSES = 3 if EVAL_MODE == '3-way' else 6

IMAGE_SIZE = 380
BATCH_SIZE = 8   # Keep small — Jupyter kernels have limited memory
DEVICE = torch.device('cpu')  # Use CPU to avoid CUDA OOM

# --- MODEL PATH (auto-selects based on mode) ---
if EVAL_MODE == '3-way':
    MODEL_PATH = 'outputs/fairface_3way/best_finetuned_model.pth'
else:
    MODEL_PATH = 'outputs/FairFace-Model-2.0/best_finetuned_model.pth'

METADATA_PATH = 'datasets/mskcc-skin-tone-labeling-dataset_metadata_2025-11-24.csv'
IMAGE_ROOT    = 'datasets/MSKCC-images'

print(f'Evaluation Mode: {EVAL_MODE}  |  NUM_CLASSES: {NUM_CLASSES}')
print(f'Model path:      {MODEL_PATH}')
print(f'Device:          {DEVICE}')
print(f'Batch size:      {BATCH_SIZE}')

In [ ]:
# ==================== CELL 2: Data Preparation ====================

print('Loading metadata...')
df = pd.read_csv(METADATA_PATH)

# 1. Keep ONLY clinical close-up images
#    Dermoscopic images use polarised light which alters perceived skin tone
df_filtered = df[df['image_type'] == 'clinical: close-up'].copy()
df_filtered = df_filtered.dropna(subset=['fitzpatrick_skin_type'])
print(f'Close-up images with skin type labels: {len(df_filtered)}')

# 2. Define Class Mappings based on EVAL_MODE
if EVAL_MODE == '3-way':
    # Training label order: 0=Dark, 1=Medium, 2=Light  (from label // 2)
    def map_gt(skin_type):
        if skin_type in ['V', 'VI']:   return 'Dark'
        if skin_type in ['III', 'IV']:  return 'Medium'
        if skin_type in ['I', 'II']:    return 'Light'
        return None

    CLASSES     = ['Dark', 'Medium', 'Light']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

    # Native 3-way model outputs (B,3) directly
    def aggregate_probs(probs):
        return probs

else:  # 6-way
    def map_gt(skin_type):
        return skin_type

    CLASSES     = ['VI', 'V', 'IV', 'III', 'II', 'I']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

    def aggregate_probs(probs_6way):
        return probs_6way

# Apply GT Mapping
df_filtered['target_label'] = df_filtered['fitzpatrick_skin_type'].apply(map_gt)
df_filtered = df_filtered.dropna(subset=['target_label'])

# Add Paths
df_filtered['path'] = df_filtered['isic_id'].apply(
    lambda x: os.path.join(IMAGE_ROOT, f'{x}.jpg')
)
df_filtered = df_filtered[df_filtered['path'].apply(os.path.exists)]
df_filtered = df_filtered.reset_index(drop=True)

print(f'Final test set size: {len(df_filtered)}')
print(f'Classes: {CLASSES}')
print(df_filtered['target_label'].value_counts())

In [ ]:
# ==================== CELL 3: Model Loading & Inference ====================

class InferenceDataset(Dataset):
    def __init__(self, df, transform):
        self.paths  = df['path'].values
        self.labels = df['target_label'].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))
        return self.transform(img), self.labels[idx]

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds = InferenceDataset(df_filtered, test_transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Build model
model = models.efficientnet_b4(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, NUM_CLASSES),
)

if os.path.exists(MODEL_PATH):
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    del state; gc.collect()
    print('✓ Model loaded from', MODEL_PATH)
else:
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

model = model.to(DEVICE).eval()

# Inference
y_true_labels = []
y_scores = []

print(f'Running inference on {len(test_ds)} images (batch_size={BATCH_SIZE})...')
with torch.no_grad():
    for i, (imgs, labels) in enumerate(test_loader):
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        batch_scores = aggregate_probs(probs)
        y_scores.extend(batch_scores)
        y_true_labels.extend(labels)
        del imgs, probs

y_scores       = np.array(y_scores)
y_true_indices = np.array([CLASS_TO_IDX[l] for l in y_true_labels])
y_pred_indices = np.argmax(y_scores, axis=1)
y_pred_labels  = [CLASSES[i] for i in y_pred_indices]

del model; gc.collect()
print('✓ Inference complete')

In [ ]:
# ==================== CELL 4: Predict & Evaluate ====================

accuracy = (y_pred_indices == y_true_indices).mean()
print(f'\n{"="*60}')
print(f'TEST SET ACCURACY: {accuracy:.2%}')
print(f'{"="*60}')

print('\nDetailed Classification Report:')
print(classification_report(y_true_indices, y_pred_indices, target_names=CLASSES))

cm = confusion_matrix(y_true_indices, y_pred_indices)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion Matrix — {EVAL_MODE}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELL 5: Precision-Recall Curves ====================

y_true_bin = label_binarize(y_true_indices, classes=list(range(NUM_CLASSES)))

# Color palette for each class
colors = plt.cm.Set1(np.linspace(0, 1, NUM_CLASSES))

fig, ax = plt.subplots(figsize=(8, 6))

for i, (cls_name, color) in enumerate(zip(CLASSES, colors)):
    prec, rec, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_scores[:, i])
    ax.plot(rec, prec, lw=2, color=color, label=f'{cls_name}  (AP = {ap:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_title(f'Precision-Recall Curves — {EVAL_MODE}', fontsize=14)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELL 6: Error Analysis (Misclassified Images) ====================

df_filtered['predicted_label'] = y_pred_labels
df_filtered['correct'] = df_filtered['target_label'] == df_filtered['predicted_label']

df_errors = df_filtered[~df_filtered['correct']].copy()
if len(df_errors) > 0:
    wrong_confs = []
    for idx, row in df_errors.iterrows():
        pred_idx = CLASS_TO_IDX.get(row['predicted_label'], 0)
        wrong_confs.append(y_scores[idx][pred_idx])
    df_errors['wrong_confidence'] = wrong_confs
    df_errors = df_errors.sort_values('wrong_confidence', ascending=False)

    n_show = min(8, len(df_errors))
    cols = max(n_show // 2, 1)
    fig, axes = plt.subplots(2, cols, figsize=(4 * cols, 8))
    axes = np.array(axes).flatten()

    for i, (_, row) in enumerate(df_errors.head(n_show).iterrows()):
        try:
            img = Image.open(row['path']).convert('RGB')
            axes[i].imshow(img)
        except Exception:
            pass
        axes[i].set_title(
            f"GT: {row['target_label']}\nPred: {row['predicted_label']}\n"
            f"Conf: {row['wrong_confidence']:.2f}", fontsize=9
        )
        axes[i].axis('off')

    plt.suptitle(f'Top {n_show} Misclassifications ({EVAL_MODE})', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No misclassifications found!')

In [ ]:
# ==================== CELL 7: Confusion Analysis per Class ====================

print('Per-class error breakdown:\n')
for i, cls in enumerate(CLASSES):
    mask_true = y_true_indices == i
    n_total   = mask_true.sum()
    n_correct = (y_pred_indices[mask_true] == i).sum()
    n_wrong   = n_total - n_correct
    acc = n_correct / n_total if n_total > 0 else 0

    if n_wrong > 0:
        wrong_preds = y_pred_indices[mask_true & (y_pred_indices != i)]
        confused_with = Counter(wrong_preds).most_common(3)
        confused_str = ', '.join(f'{CLASSES[c]}: {n}' for c, n in confused_with)
    else:
        confused_str = 'N/A'

    print(f'  {cls:>10s}  |  Support: {n_total:4d}  |  Acc: {acc:.1%}  |  Confused with: {confused_str}')

In [ ]:
# ==================== CELL 8: Accuracy by Fitzpatrick Type ====================

print('Accuracy broken down by original Fitzpatrick skin type:\n')

fitz_types = sorted(df_filtered['fitzpatrick_skin_type'].unique())
results = []
for ft in fitz_types:
    mask = df_filtered['fitzpatrick_skin_type'] == ft
    subset_true = y_true_indices[mask]
    subset_pred = y_pred_indices[mask]
    n = mask.sum()
    acc = (subset_true == subset_pred).mean() if n > 0 else 0
    results.append({'Fitzpatrick': ft, 'Support': int(n), 'Accuracy': f'{acc:.1%}'})

print(pd.DataFrame(results).to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
accs = [float(r['Accuracy'].strip('%')) / 100 for r in results]
bars = ax.bar([r['Fitzpatrick'] for r in results], accs, color=plt.cm.RdYlGn(accs))
ax.set_ylabel('Accuracy')
ax.set_xlabel('Fitzpatrick Type')
ax.set_title(f'Accuracy by Fitzpatrick Type ({EVAL_MODE})')
ax.set_ylim(0, 1)
for bar, a in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{a:.1%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()